In [1]:
import json
import os
import dotenv

import numpy as np
import pandas as pd

from sklearn.cluster import HDBSCAN

import sqlalchemy as sa
from sqlalchemy import select, create_engine

from models import Wards, Matches, Players

dotenv.load_dotenv()

True

In [2]:
db = create_engine(os.environ['DB_URL'])

In [3]:
df = pd.DataFrame(pd.read_sql(
    sql=select(Wards, Matches, Players).join(Players, Wards.player_placed).join(Matches).where(sa.and_(Wards.is_obs, Matches.league_id == 15728)).order_by(Matches.id.desc()),
    con=db
))

In [4]:
df['duration'] = df[['duration', 'duration_1', 'time_placed']].apply((lambda row: min(row['duration'], row['duration_1'] - row['time_placed'])), axis=1)

In [5]:
hdbscan = HDBSCAN(
    cluster_selection_epsilon=70,
    min_samples=5,
    min_cluster_size=5,
    cluster_selection_method='eom',
    n_jobs=-1,
)
hdbscan.fit(df[['x_pos', 'y_pos']])

df['cluster_id'] = hdbscan.labels_

In [6]:
df.groupby('cluster_id').size().sort_values(ascending=False)

cluster_id
-1      641
 57     235
 35     211
 4      151
 3      150
       ... 
 101      5
 147      5
 64       5
 157      5
 79       5
Length: 177, dtype: int64

In [7]:
df['advantage'] = df[['is_radiant', 'radiant_networth', 'dire_networth']].apply(lambda row: (row['radiant_networth'] - row['dire_networth']) if row['is_radiant'] else (row['dire_networth'] - row['radiant_networth']), axis=1)
df

,id,match_id,is_obs,is_radiant,is_destroyed,is_post_game,player_placed_id,player_destroyed_id,time_placed,duration,...,id_1,league_id,duration_1,region,date,game_version,id_2,name,cluster_id,advantage
0,837950,7406531302,True,True,False,False,16497807,NaN,1181,360,...,7406531302,15728,2525,1,1698621223,53,16497807,tOfu,57,-1973
1,837949,7406531302,True,False,False,False,256156323,NaN,1179,360,...,7406531302,15728,2525,1,1698621223,53,256156323,Mira,19,1881
2,837948,7406531302,True,False,False,False,113331514,NaN,1139,360,...,7406531302,15728,2525,1,1698621223,53,113331514,Miposhka,100,1699
3,837944,7406531302,True,False,False,False,113331514,NaN,1053,360,...,7406531302,15728,2525,1,1698621223,53,113331514,Miposhka,164,1960
4,837942,7406531302,True,False,False,False,113331514,NaN,1031,360,...,7406531302,15728,2525,1,1698621223,53,113331514,Miposhka,35,1960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5366,827144,7378530387,True,True,True,False,10366616,136829091.0,703,338,...,7378530387,15728,1405,1,1697130044,53,10366616,Sneyking,109,3176
5367,827135,7378530387,True,False,False,False,346412363,NaN,457,360,...,7378530387,15728,1405,1,1697130044,53,346412363,Ari,-1,-1985
5368,827134,7378530387,True,False,False,False,136829091,NaN,357,360,...,7378530387,15728,1405,1,1697130044,53,136829091,Whitemon,46,-1548
5369,827133,7378530387,True,True,False,False,10366616,NaN,354,360,...,7378530387,15728,1405,1,1697130044,53,10366616,Sneyking,148,1548


In [8]:
grouped = df.groupby(['cluster_id', 'is_radiant']).agg({
        'x_pos': lambda x: round(x.mean()),
        'y_pos': lambda x: round(x.mean()),
        'z_pos': lambda x: x.mode().iloc[0], #- 16384,
        'duration': lambda x: round(x.mean()),
        'time_placed': lambda x: round(x.mean()),
        'advantage': lambda x: round(x.mean()),
        'is_destroyed': 'sum'
    })
grouped['amount'] = df.groupby(['cluster_id', 'is_radiant']).size()
# grouped

In [9]:
players = df[df['cluster_id'] == 1][['player_placed_id', 'name']].groupby(['player_placed_id', 'name']).size().to_frame('amount').reset_index().to_dict(orient='records')
players

[{'player_placed_id': 16497807, 'name': 'tOfu', 'amount': 1},
 {'player_placed_id': 34505203, 'name': 'MinD_ContRoL', 'amount': 2},
 {'player_placed_id': 54580962, 'name': 'iNSaNiA', 'amount': 2},
 {'player_placed_id': 77490514, 'name': 'Boxi', 'amount': 2},
 {'player_placed_id': 81475303, 'name': 'Yamsun', 'amount': 2},
 {'player_placed_id': 85696663, 'name': 'n1ght', 'amount': 1},
 {'player_placed_id': 91730177, 'name': 'Seleri', 'amount': 2},
 {'player_placed_id': 94049589, 'name': 'fng', 'amount': 3},
 {'player_placed_id': 97590558, 'name': 'Ace', 'amount': 1},
 {'player_placed_id': 101259972, 'name': 'Oli~', 'amount': 5},
 {'player_placed_id': 111114687, 'name': 'y`', 'amount': 2},
 {'player_placed_id': 113331514, 'name': 'Miposhka', 'amount': 1},
 {'player_placed_id': 117089283, 'name': 'Pandaboo', 'amount': 1},
 {'player_placed_id': 127565532, 'name': 'Fishman', 'amount': 1},
 {'player_placed_id': 134556694, 'name': 'Solo', 'amount': 3},
 {'player_placed_id': 136829091, 'name': 

In [10]:
players_destroyed = df[df['cluster_id'] != -1].groupby(['cluster_id', 'player_destroyed_id']).agg({'duration': lambda x: round(x.mean())})
players_destroyed['amount'] = df.groupby(['cluster_id', 'player_destroyed_id']).size()
players_destroyed

duration  amount
cluster_id player_destroyed_id                  
0          10366616.0                 42       1
           16497807.0                 37       1
           25907144.0                105       2
           77490514.0                106       3
           81475303.0                108       2
...                                  ...     ...
175        54580962.0                 98       1
           87063175.0                126       1
           101695162.0               150       1
           161839895.0                56       1
           317880638.0               193       1

[1294 rows x 2 columns]

In [11]:
for i in range(len(set(hdbscan.labels_)) - 1):
    current = grouped.loc[i]

    d = {
        'cluster_id': i,
        'x_pos': round(current['x_pos'].mean()),
        'y_pos': round(current['y_pos'].mean()),
        'z_pos': round(current['z_pos'].mean()),
        'radiant': {
            'amount': current.loc[True]['amount'],
            'destroyed': current.loc[True]['is_destroyed'],
            'advantage': current.loc[True]['advantage'],
            'duration': current.loc[True]['duration'],
            'time_placed': current.loc[True]['time_placed']
        } if True in current.index else None,
        'dire': {
            'amount': current.loc[False]['amount'],
            'destroyed': current.loc[False]['is_destroyed'],
            'advantage': current.loc[False]['advantage'],
            'duration': current.loc[False]['duration'],
            'time_placed': current.loc[False]['time_placed']
        } if False in current.index else None,
        'all': {
            'amount': current['amount'].sum(),
            'destroyed': current['is_destroyed'].sum(),
            'advantage': None,
            'duration': round(current['duration'].mean()),
            'time_placed': round(current['time_placed'].mean())
        }
    }

    print(json.dumps(d, indent=4, default=int))

{
    "cluster_id": 0,
    "x_pos": 14834,
    "y_pos": 23287,
    "z_pos": 16784,
    "radiant": {
        "amount": 48,
        "destroyed": 30,
        "advantage": 1528,
        "duration": 193,
        "time_placed": 1522
    },
    "dire": {
        "amount": 44,
        "destroyed": 15,
        "advantage": -801,
        "duration": 285,
        "time_placed": 1697
    },
    "all": {
        "amount": 92,
        "destroyed": 45,
        "advantage": null,
        "duration": 239,
        "time_placed": 1610
    }
}
{
    "cluster_id": 1,
    "x_pos": 8453,
    "y_pos": 18196,
    "z_pos": 16920,
    "radiant": {
        "amount": 23,
        "destroyed": 8,
        "advantage": -2683,
        "duration": 270,
        "time_placed": 1991
    },
    "dire": {
        "amount": 27,
        "destroyed": 8,
        "advantage": 77,
        "duration": 289,
        "time_placed": 1661
    },
    "all": {
        "amount": 50,
        "destroyed": 16,
        "advantage": null,
     

In [12]:
average_df = grouped.reset_index('is_radiant')
average_radiant = average_df[average_df['is_radiant'] == True]
average_dire = average_df[average_df['is_radiant'] == False]
average = {
    'cluster_id': -1,
    'radiant': {
        'advantage': round(average_radiant['advantage'].mean()),
        'duration': round(average_radiant['duration'].mean()),
    },
    'dire': {
        'advantage': round(average_dire['advantage'].mean()),
        'duration': round(average_dire['duration'].mean()),
    },
    'all': {
        'advantage': None,
        'duration': round(average_df['duration'].mean()),
    }
}
average

{'cluster_id': -1,
 'radiant': {'advantage': -954, 'duration': 284},
 'dire': {'advantage': 322, 'duration': 273},
 'all': {'advantage': None, 'duration': 279}}